# Whisper Medium LoRA: 아동 발화 학습 + ASD 소규모 적응

이 노트북은 일반 아동 발화로 Whisper Medium LoRA를 학습한 뒤, ASD 5개 발화와 일반 아동 일부를 섞어 단일 화자 적응 실험을 수행합니다. ASD 2개 발화는 학습에서 제외하고 정성 평가에 사용합니다.

> ASD 데이터가 한 화자뿐이므로 결과를 ASD 아동 전체에 일반화할 수 없습니다. 이 실험은 프로토타입 및 개인화 가능성 검증용입니다.

In [ ]:
# Colab이 제공하는 CUDA용 torch/torchaudio는 재설치하지 않습니다.
%pip install -q 'transformers>=4.46,<5' 'datasets>=3,<5' 'accelerate>=1,<2' 'peft>=0.13,<1' 'evaluate>=0.4,<1' 'jiwer>=3,<5' 'librosa>=0.10,<1' 'soundfile>=0.12,<1' 'PyYAML>=6,<7' 'tensorboard>=2.17,<3' 'sentencepiece>=0.2,<1'

In [ ]:
import os
import random
import re
import subprocess
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import jiwer
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import yaml
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

assert torch.cuda.is_available(), 'GPU 런타임을 선택해주세요: Select Kernel > Colab > GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

In [ ]:
# Colab VM에는 로컬 VS Code 저장소가 자동으로 복사되지 않으므로 원격 브랜치를 clone합니다.
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/Mazu-Talk/MazuTalk.git'
BRANCH = 'feat/whisper-medium-lora-training'
REPO_DIR = Path('/content/MazuTalk')

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
CONFIG_PATH = REPO_DIR / 'ai/stt/configs/whisper_medium_lora.yaml'
with CONFIG_PATH.open(encoding='utf-8') as file:
    config = yaml.safe_load(file)

config

## 데이터 압축 해제

Google Drive의 `MazuTalk/stt_data/mazutalk_stt_data_10k.tar`를 Colab VM으로 해제합니다. 압축 파일 내부에는 저장소 기준 상대 경로인 `ai/data/...`, `ai/stt/data/...`가 유지되어야 합니다.

In [ ]:
archive_path = Path(config['data']['archive_path'])
data_root = Path(config['data']['runtime_root'])
child_metadata_path = data_root / config['data']['child_metadata']
asd_metadata_path = data_root / config['data']['asd_metadata']

if not child_metadata_path.exists() or not asd_metadata_path.exists():
    if not archive_path.exists():
        raise FileNotFoundError(
            f'{archive_path}가 없습니다. 로컬에서 데이터 tar를 만든 뒤 Google Drive에 업로드해주세요.'
        )
    data_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xf', str(archive_path), '-C', str(data_root)], check=True)

assert child_metadata_path.exists(), child_metadata_path
assert asd_metadata_path.exists(), asd_metadata_path
print('child metadata:', child_metadata_path)
print('ASD metadata:', asd_metadata_path)

In [ ]:
SEED = int(config['split']['seed'])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def resolve_audio_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else data_root / path

def prepare_frame(frame):
    result = frame.copy()
    result['text'] = result['text'].fillna('').astype(str).str.strip()
    result['speaker_id'] = result['speaker_id'].fillna('unknown').astype(str)
    result['resolved_audio_path'] = result['audio_path'].map(resolve_audio_path)
    result = result[result['text'].ne('')]
    result = result[result['duration_sec'].astype(float).le(float(config['data']['max_audio_seconds']))]
    result = result[result['resolved_audio_path'].map(Path.exists)]
    return result.reset_index(drop=True)

child = prepare_frame(pd.read_csv(child_metadata_path))
asd = prepare_frame(pd.read_csv(asd_metadata_path))

speakers = child['speaker_id'].drop_duplicates().to_numpy()
rng = np.random.default_rng(SEED)
rng.shuffle(speakers)
n_val = max(1, int(len(speakers) * float(config['split']['validation_ratio'])))
n_test = max(1, int(len(speakers) * float(config['split']['test_ratio'])))
val_speakers = set(speakers[:n_val])
test_speakers = set(speakers[n_val:n_val + n_test])
train_speakers = set(speakers[n_val + n_test:])

child_train = child[child['speaker_id'].isin(train_speakers)]
child_val = child[child['speaker_id'].isin(val_speakers)]
child_test = child[child['speaker_id'].isin(test_speakers)]

train_limit = int(config['data']['child_train_limit'])
eval_limit = int(config['data']['child_eval_limit'])
child_train = child_train.sample(min(train_limit, len(child_train)), random_state=SEED).reset_index(drop=True)
child_val = child_val.sample(min(eval_limit, len(child_val)), random_state=SEED).reset_index(drop=True)
child_test = child_test.sample(min(eval_limit, len(child_test)), random_state=SEED).reset_index(drop=True)

asd_names = asd['resolved_audio_path'].map(lambda path: Path(path).name)
asd_train = asd[asd_names.isin(config['asd_adaptation']['train_files'])].reset_index(drop=True)
asd_test = asd[asd_names.isin(config['asd_adaptation']['test_files'])].reset_index(drop=True)

print({'child_train': len(child_train), 'child_val': len(child_val), 'child_test': len(child_test)})
print({'asd_train': len(asd_train), 'asd_test': len(asd_test)})
display(asd[['audio_path', 'text']])

In [ ]:
MODEL_ID = config['model']['id']
LANGUAGE = config['model']['language']
TASK = config['model']['task']
SAMPLE_RATE = int(config['data']['sample_rate'])

processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

class SpeechDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        audio, sample_rate = sf.read(row['resolved_audio_path'], dtype='float32', always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sample_rate != SAMPLE_RATE:
            audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
        input_features = processor.feature_extractor(
            audio, sampling_rate=SAMPLE_RATE, return_tensors='pt'
        ).input_features[0]
        labels = processor.tokenizer(str(row['text'])).input_ids
        return {'input_features': input_features, 'labels': labels}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor

    def __call__(self, features):
        input_features = [{'input_features': item['input_features']} for item in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': item['labels']} for item in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch['attention_mask'].ne(1), -100)
        decoder_start = self.processor.tokenizer.convert_tokens_to_ids('<|startoftranscript|>')
        if labels.shape[1] and (labels[:, 0] == decoder_start).all():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)
child_train_ds = SpeechDataset(child_train)
child_val_ds = SpeechDataset(child_val)
child_test_ds = SpeechDataset(child_test)
asd_train_ds = SpeechDataset(asd_train)
asd_test_ds = SpeechDataset(asd_test)

In [ ]:
def normalize_korean(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    return re.sub(r'[\s\W_]+', '', text, flags=re.UNICODE)

def compute_metrics(prediction):
    pred_ids = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
    label_ids = np.where(
        prediction.label_ids == -100,
        processor.tokenizer.pad_token_id,
        prediction.label_ids,
    )
    predictions = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    references = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    normalized_predictions = [normalize_korean(text) for text in predictions]
    normalized_references = [normalize_korean(text) for text in references]
    return {
        'cer': jiwer.cer(references, predictions),
        'cer_normalized': jiwer.cer(normalized_references, normalized_predictions),
    }

device = torch.device('cuda')
dtype = torch.float16
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
)
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False
model.to(device)

In [ ]:
@torch.inference_mode()
def transcribe_frame(current_model, frame, variant, prompt=None):
    dataset = SpeechDataset(frame)
    rows = []
    current_model.eval()
    for index in range(len(dataset)):
        item = dataset[index]
        generate_args = {
            'language': LANGUAGE,
            'task': TASK,
            'num_beams': int(config['prompt']['num_beams']),
            'max_new_tokens': int(config['model']['max_label_length']),
        }
        if prompt:
            generate_args['prompt_ids'] = processor.get_prompt_ids(prompt, return_tensors='pt').to(device)
        generated = current_model.generate(
            input_features=item['input_features'].unsqueeze(0).to(device=device, dtype=dtype),
            **generate_args,
        )
        prediction = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()
        reference = str(frame.iloc[index]['text'])
        rows.append({
            'variant': variant,
            'audio_path': str(frame.iloc[index]['audio_path']),
            'reference': reference,
            'prediction': prediction,
            'cer_normalized': jiwer.cer(normalize_korean(reference), normalize_korean(prediction)),
        })
    return pd.DataFrame(rows)

baseline_asd = transcribe_frame(model, asd_test, 'base_medium')
display(baseline_asd)

## 1단계: 일반 아동 발화 LoRA 학습

기본 설정은 Colab에서 먼저 정상 동작을 확인하기 위한 5,000개/500 step 파일럿입니다. 전체 59,108개 학습은 설정 파일의 `child_train_limit`과 `max_steps`를 늘려 별도 실행합니다.

In [ ]:
lora_config = LoraConfig(
    r=int(config['lora']['rank']),
    lora_alpha=int(config['lora']['alpha']),
    lora_dropout=float(config['lora']['dropout']),
    bias='none',
    target_modules=list(config['lora']['target_modules']),
    task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.print_trainable_parameters()

train_config = config['training']
output_dir = Path(train_config['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(output_dir / 'child'),
    per_device_train_batch_size=int(train_config['train_batch_size']),
    per_device_eval_batch_size=int(train_config['eval_batch_size']),
    gradient_accumulation_steps=int(train_config['gradient_accumulation_steps']),
    learning_rate=float(train_config['learning_rate']),
    max_steps=int(train_config['max_steps']),
    warmup_steps=int(train_config['warmup_steps']),
    fp16=bool(train_config['fp16']),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='steps',
    save_strategy='steps',
    logging_steps=int(train_config['logging_steps']),
    eval_steps=int(train_config['eval_steps']),
    save_steps=int(train_config['save_steps']),
    save_total_limit=int(train_config['save_total_limit']),
    predict_with_generate=True,
    generation_max_length=int(train_config['generation_max_length']),
    load_best_model_at_end=True,
    metric_for_best_model='cer_normalized',
    greater_is_better=False,
    remove_unused_columns=False,
    label_names=['labels'],
    report_to=['tensorboard'],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=child_train_ds,
    eval_dataset=child_val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.save_model(str(output_dir / 'adapter-child'))
processor.save_pretrained(str(output_dir / 'adapter-child'))

In [ ]:
child_metrics = trainer.evaluate(child_test_ds, metric_key_prefix='child_test')
child_lora_asd = transcribe_frame(model, asd_test, 'child_lora')
print(child_metrics)
display(child_lora_asd)

## 2단계: ASD 단일 화자 소규모 적응

ASD 5개를 그대로 반복하는 대신 일반 아동 표본을 함께 넣어 기존 성능 손상을 줄입니다. 데이터가 매우 적으므로 50 step만 수행하며 결과는 일반화가 아닌 개인화 실험으로 해석합니다.

In [ ]:
if config['asd_adaptation']['enabled']:
    retention_count = min(int(config['data']['child_retention_samples']), len(child_train))
    child_retention = child_train.sample(retention_count, random_state=SEED)
    repeated_asd = pd.concat(
        [asd_train] * int(config['asd_adaptation']['repeat_factor']),
        ignore_index=True,
    )
    adaptation_frame = pd.concat([child_retention, repeated_asd], ignore_index=True)
    adaptation_frame = adaptation_frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
    adaptation_ds = SpeechDataset(adaptation_frame)

    adaptation_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir / 'asd-adaptation'),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=int(train_config['gradient_accumulation_steps']),
        learning_rate=float(config['asd_adaptation']['learning_rate']),
        max_steps=int(config['asd_adaptation']['max_steps']),
        warmup_steps=int(config['asd_adaptation']['warmup_steps']),
        fp16=bool(train_config['fp16']),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        eval_strategy='steps',
        save_strategy='steps',
        logging_steps=5,
        eval_steps=25,
        save_steps=25,
        save_total_limit=2,
        predict_with_generate=True,
        generation_max_length=int(train_config['generation_max_length']),
        load_best_model_at_end=True,
        metric_for_best_model='cer_normalized',
        greater_is_better=False,
        remove_unused_columns=False,
        label_names=['labels'],
        report_to=['tensorboard'],
    )
    adaptation_trainer = Seq2SeqTrainer(
        model=model,
        args=adaptation_args,
        train_dataset=adaptation_ds,
        eval_dataset=asd_test_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    adaptation_trainer.train()
    adaptation_trainer.save_model(str(output_dir / 'adapter-child-asd'))
    processor.save_pretrained(str(output_dir / 'adapter-child-asd'))

In [ ]:
prompt_text = config['prompt']['text']
adapted_asd = transcribe_frame(model, asd_test, 'child_asd_lora')
prompted_asd = transcribe_frame(model, asd_test, 'child_asd_lora_prompt', prompt=prompt_text)
comparison = pd.concat([baseline_asd, child_lora_asd, adapted_asd, prompted_asd], ignore_index=True)
results_path = output_dir / 'asd_comparison.csv'
comparison.to_csv(results_path, index=False, encoding='utf-8-sig')
display(comparison)
display(comparison.groupby('variant')['cer_normalized'].mean().sort_values())
print('saved:', results_path)

## Adapter 병합 및 Faster-Whisper 변환

아래 셀은 학습과 평가가 끝난 뒤 한 번만 실행합니다. 병합 후에는 추가 LoRA 학습을 계속할 수 없으므로 체크포인트가 Drive에 저장되었는지 먼저 확인하세요.

In [ ]:
merged_dir = output_dir / 'merged-transformers'
merged_model = model.merge_and_unload()
merged_model.config.use_cache = True
merged_model.save_pretrained(merged_dir, safe_serialization=True)
processor.save_pretrained(merged_dir)
print('merged model:', merged_dir)
print('다음 단계: ctranslate2 설치 후 ct2-transformers-converter로 변환하세요.')

In [ ]:
# 선택 실행: Faster-Whisper가 읽을 CTranslate2 float16 모델 생성
%pip install -q ctranslate2
ct2_dir = output_dir / 'faster-whisper-ct2-float16'
subprocess.run([
    'ct2-transformers-converter',
    '--model', str(merged_dir),
    '--output_dir', str(ct2_dir),
    '--quantization', 'float16',
    '--force',
], check=True)
print('Faster-Whisper model:', ct2_dir)